# Q1 Regression：Implicit Neural Representation (INR) of an Image

**姓名 / 學號：** ＿＿＿＿＿＿＿＿

本作業以 **NumPy 手刻** 深度神經網路（forward、backpropagation、mini-batch SGD），不使用任何現成的 ML / DL 套件。
網路學習函數 $f_w(x, y) \rightarrow (r, g, b)$，將 2D 座標映射到 RGB 顏色。

- 使用套件：`numpy`（運算）、`PIL`（僅讀圖）、`matplotlib`（僅畫圖）
- 實驗 A：直接輸入原始座標 $(x, y)$
- 實驗 B：輸入經 Fourier feature mapping 編碼的座標

> 執行方式：由上往下依序執行（或 Run All）。Cell 7、8 為訓練，需要一些時間，請勿重複執行。

## 0. 環境設定與超參數

In [ ]:
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.random.seed(42)

IMG_PATH = "2026_NYCU.png"
IMG_SIZE = 100

EPOCHS = 800
BATCH_SIZE = 256
LEARNING_RATE = 0.005      # 純 SGD 若 loss 幾乎不動，可試 0.05 / 0.1 / 0.5

NUM_FREQ = 6
HIDDEN_UNITS = 128
SNAPSHOT_EPOCHS = [100, 200, 400, 800]

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. 讀取圖片與前處理

- RGB 由 $[0, 255]$ 正規化到 $[0, 1]$
- 若圖片含透明背景，先貼到白底，避免透明處被轉成黑色

In [ ]:
if not os.path.exists(IMG_PATH):
    raise FileNotFoundError(f"找不到圖片：{IMG_PATH}（請放在與 notebook 相同的資料夾）")

img_raw = Image.open(IMG_PATH).convert("RGBA")
background = Image.new("RGBA", img_raw.size, (255, 255, 255, 255))
img = Image.alpha_composite(background, img_raw).convert("RGB")
img = img.resize((IMG_SIZE, IMG_SIZE))

# RGB values: [0, 255] -> [0, 1]
img_arr = np.asarray(img, dtype=np.float64) / 255.0

H, W, C = img_arr.shape
N = H * W

print("Image shape:", img_arr.shape)
print("Number of pixels:", N)

plt.imshow(img_arr)
plt.title("Ground Truth Image")
plt.axis("off")
plt.show()

## 2. 建立正規化座標網格

$x, y \in [-1, 1]$，攤平後座標為 $(N, 2)$，顏色為 $(N, 3)$，並保留原始的 raster 順序以便之後重建影像。

In [ ]:
x = np.linspace(-1.0, 1.0, W)
y = np.linspace(-1.0, 1.0, H)

xx, yy = np.meshgrid(x, y, indexing="xy")

# Preserve the original raster order for reconstruction.
full_coords = np.stack(
    [xx.reshape(-1), yy.reshape(-1)],
    axis=1
)

# Each pixel has one RGB target.
colors = img_arr.reshape(-1, 3)

print("Coordinates shape:", full_coords.shape)
print("Colors shape:", colors.shape)

## 3. Shuffle 並切分：90% 訓練、10% 測試

In [ ]:
indices = np.random.permutation(N)
n_train = int(0.9 * N)

train_idx = indices[:n_train]
test_idx = indices[n_train:]

X_train = full_coords[train_idx]
Y_train = colors[train_idx]

X_test = full_coords[test_idx]
Y_test = colors[test_idx]

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_test :", X_test.shape)
print("Y_test :", Y_test.shape)

## 4. Fourier Feature Mapping

$$
\gamma(\mathbf{p}) = \big[\, \mathbf{p},\ \sin(2^0\pi \mathbf{p}),\ \cos(2^0\pi \mathbf{p}),\ \dots,\ \sin(2^{K-1}\pi \mathbf{p}),\ \cos(2^{K-1}\pi \mathbf{p}) \,\big]
$$

其中 $\mathbf{p} = (x, y)$，$K$ = `NUM_FREQ` = 6，輸出維度 = $2 + 4K = 26$。

In [ ]:
def fourier_features(x, num_freq=6):
    """
    Input:
        x: (N, 2), normalized coordinates

    Output:
        (N, 2 + 4*num_freq)

    Features include raw coordinates and sine/cosine
    features for both x and y coordinates.
    """
    freqs = (2.0 ** np.arange(num_freq)) * np.pi

    features = [x]

    for freq in freqs:
        features.append(np.sin(freq * x))
        features.append(np.cos(freq * x))

    return np.concatenate(features, axis=1)

## 5. 手刻 MLP（Forward / Backpropagation / SGD）

- 隱藏層：ReLU；輸出層：Sigmoid（讓輸出落在 $[0,1]$）
- 初始化：He initialization
- 損失函數：MSE（對樣本與 RGB 三通道取平均）

**Backpropagation 推導**（$m$ 為 batch size，輸出維度 3）：

$$
\frac{\partial E}{\partial \mathbf{a}^{(L)}} = \frac{2(\mathbf{a}^{(L)} - \mathbf{t})}{3m},\qquad
\boldsymbol{\delta}^{(L)} = \frac{\partial E}{\partial \mathbf{a}^{(L)}} \odot \mathbf{a}^{(L)}\odot(1-\mathbf{a}^{(L)})
$$

$$
\boldsymbol{\delta}^{(l)} = \big(\boldsymbol{\delta}^{(l+1)} (W^{(l+1)})^\top\big) \odot \mathbb{1}[\mathbf{z}^{(l)} > 0]
$$

$$
\nabla_{W^{(l)}} E = (\mathbf{a}^{(l-1)})^\top \boldsymbol{\delta}^{(l)},\qquad
\nabla_{b^{(l)}} E = \sum_i \boldsymbol{\delta}^{(l)}_i
$$

**SGD 更新：** $W \leftarrow W - \eta\,\nabla_W E,\quad b \leftarrow b - \eta\,\nabla_b E$

In [ ]:
class MLP:
    def __init__(self, layer_sizes, lr=0.01):
        self.lr = lr
        self.L = len(layer_sizes) - 1

        self.W = []
        self.b = []

        for i in range(self.L):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i + 1]

            # He initialization
            weight = (
                np.random.randn(fan_in, fan_out)
                * np.sqrt(2.0 / fan_in)
            )

            bias = np.zeros((1, fan_out))

            self.W.append(weight)
            self.b.append(bias)

    @staticmethod
    def relu(z):
        return np.maximum(0.0, z)

    @staticmethod
    def relu_grad(z):
        return (z > 0).astype(np.float64)

    @staticmethod
    def sigmoid(z):
        # Clip for numerical stability.
        z = np.clip(z, -500.0, 500.0)
        return 1.0 / (1.0 + np.exp(-z))

    def forward(self, X):
        self.z_cache = []
        self.a_cache = [X]

        a = X

        for i in range(self.L):
            z = a @ self.W[i] + self.b[i]

            if i < self.L - 1:
                a = self.relu(z)
            else:
                a = self.sigmoid(z)

            self.z_cache.append(z)
            self.a_cache.append(a)

        return a

    def backward(self, Y):
        """
        Backpropagation for mean squared error:

        MSE = mean((prediction - target) ** 2)

        The mean is over both samples and RGB channels.
        """
        m = Y.shape[0]
        output_dim = Y.shape[1]

        y_pred = self.a_cache[-1]

        # dMSE / dA_output
        dA = 2.0 * (y_pred - Y) / (m * output_dim)

        # Sigmoid derivative
        dZ = dA * y_pred * (1.0 - y_pred)

        grads_W = [None] * self.L
        grads_b = [None] * self.L

        for i in reversed(range(self.L)):
            A_prev = self.a_cache[i]

            grads_W[i] = A_prev.T @ dZ
            grads_b[i] = np.sum(
                dZ, axis=0, keepdims=True
            )

            if i > 0:
                dA_prev = dZ @ self.W[i].T
                dZ = (
                    dA_prev
                    * self.relu_grad(self.z_cache[i - 1])
                )

        # SGD parameter update
        for i in range(self.L):
            self.W[i] -= self.lr * grads_W[i]
            self.b[i] -= self.lr * grads_b[i]

    def predict(self, X):
        return self.forward(X)

## 6. 評估指標與訓練迴圈（mini-batch SGD）

$$
E_{\text{RMS}} = \sqrt{\frac{1}{3N}\sum_{n=1}^{N}\sum_{c\in\{R,G,B\}} (t_{nc} - y_c(\mathbf{x}_n;\mathbf{w}))^2}
$$

每個 epoch 重新 shuffle 訓練資料，再以 mini-batch 逐批更新；每個 epoch 結束時記錄 train / test MSE，並在指定 epoch 用「全部座標」重建影像。

In [ ]:
def calculate_mse(model, X, Y):
    prediction = model.predict(X)
    return float(np.mean((prediction - Y) ** 2))


def calculate_rms(model, X, Y):
    return float(np.sqrt(calculate_mse(model, X, Y)))


def train_model(
    X_train,
    Y_train,
    X_test,
    Y_test,
    layer_sizes,
    lr,
    epochs,
    batch_size,
    snapshot_epochs,
    full_coords
):
    model = MLP(layer_sizes, lr=lr)

    n = X_train.shape[0]

    train_mse_history = []
    test_mse_history = []
    snapshots = {}

    for epoch in range(1, epochs + 1):

        # Shuffle training samples at each epoch.
        perm = np.random.permutation(n)
        X_shuffled = X_train[perm]
        Y_shuffled = Y_train[perm]

        # Mini-batch updates.
        for start in range(0, n, batch_size):
            end = start + batch_size

            xb = X_shuffled[start:end]
            yb = Y_shuffled[start:end]

            model.forward(xb)
            model.backward(yb)

        # Evaluate after all updates in this epoch.
        train_mse = calculate_mse(model, X_train, Y_train)
        test_mse = calculate_mse(model, X_test, Y_test)

        train_mse_history.append(train_mse)
        test_mse_history.append(test_mse)

        # Save reconstruction in original image order.
        if epoch in snapshot_epochs:
            reconstruction = model.predict(full_coords)
            snapshots[epoch] = reconstruction.copy()

        if (
            epoch == 1
            or epoch % 50 == 0
            or epoch in snapshot_epochs
        ):
            train_rms = np.sqrt(train_mse)
            test_rms = np.sqrt(test_mse)

            print(
                f"Epoch {epoch:4d}/{epochs} | "
                f"Train MSE: {train_mse:.6f} | "
                f"Test MSE: {test_mse:.6f} | "
                f"Train RMS: {train_rms:.6f} | "
                f"Test RMS: {test_rms:.6f}"
            )

    return (
        model,
        train_mse_history,
        test_mse_history,
        snapshots
    )

## (a) 目標函數與評估指標

- **訓練目標：** 最小化 MSE，$E(\mathbf{w}) = \frac{1}{3m}\sum \lVert \mathbf{t}_n - \mathbf{y}(\mathbf{x}_n;\mathbf{w}) \rVert^2$（與題目的 SSE 僅差一個常數倍，最佳解相同）
- **評估指標：** 分別在訓練集與測試集計算 RMS error（公式見第 6 節，`calculate_rms`）
- **最佳化：** 手刻 backpropagation + mini-batch SGD，未使用任何現成最佳化函式

## (b) 實驗結果

### 實驗 A：Raw coordinates $(x, y)$

In [ ]:
print("========== RAW COORDINATES ==========")

raw_layer_sizes = [
    2,
    HIDDEN_UNITS,
    HIDDEN_UNITS,
    HIDDEN_UNITS,
    3
]

model_raw, loss_raw, test_loss_raw, snap_raw = train_model(
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    Y_test=Y_test,
    layer_sizes=raw_layer_sizes,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    snapshot_epochs=SNAPSHOT_EPOCHS,
    full_coords=full_coords
)

train_rms_raw = calculate_rms(model_raw, X_train, Y_train)
test_rms_raw = calculate_rms(model_raw, X_test, Y_test)

print("\nRaw-coordinate final results")
print("Train RMS:", train_rms_raw)
print("Test RMS :", test_rms_raw)

### 實驗 B：Fourier-encoded coordinates

In [ ]:
print("========== FOURIER ENCODING ==========")

X_train_enc = fourier_features(X_train, NUM_FREQ)
X_test_enc = fourier_features(X_test, NUM_FREQ)
full_coords_enc = fourier_features(full_coords, NUM_FREQ)

enc_dim = X_train_enc.shape[1]

print("Fourier feature dimension:", enc_dim)

enc_layer_sizes = [
    enc_dim,
    HIDDEN_UNITS,
    HIDDEN_UNITS,
    HIDDEN_UNITS,
    3
]

model_enc, loss_enc, test_loss_enc, snap_enc = train_model(
    X_train=X_train_enc,
    Y_train=Y_train,
    X_test=X_test_enc,
    Y_test=Y_test,
    layer_sizes=enc_layer_sizes,
    lr=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    snapshot_epochs=SNAPSHOT_EPOCHS,
    full_coords=full_coords_enc
)

train_rms_enc = calculate_rms(model_enc, X_train_enc, Y_train)
test_rms_enc = calculate_rms(model_enc, X_test_enc, Y_test)

print("\nFourier-encoded final results")
print("Train RMS:", train_rms_enc)
print("Test RMS :", test_rms_enc)

### (b)-(1) 網路架構，(b)-(3)(4) Train / Test RMS

In [ ]:
print("===== Network Architecture =====")
print("Raw     :", " -> ".join(map(str, raw_layer_sizes)),
      "(ReLU x3 hidden, Sigmoid output)")
print("Fourier :", " -> ".join(map(str, enc_layer_sizes)),
      "(ReLU x3 hidden, Sigmoid output)")
print(f"Hidden layers: 3 | Units/layer: {HIDDEN_UNITS} | "
      f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")

print("\n===== RMS Error =====")
print(f"{'Input':<10}{'Train RMS':>12}{'Test RMS':>12}")
print(f"{'Raw':<10}{train_rms_raw:>12.4f}{test_rms_raw:>12.4f}")
print(f"{'Fourier':<10}{train_rms_enc:>12.4f}{test_rms_enc:>12.4f}")

### (b)-(2) Learning Curve

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(range(1, EPOCHS + 1), loss_raw, label="Raw Train MSE")
plt.plot(range(1, EPOCHS + 1), test_loss_raw, label="Raw Test MSE")
plt.plot(range(1, EPOCHS + 1), loss_enc, label="Fourier Train MSE")
plt.plot(range(1, EPOCHS + 1), test_loss_enc, label="Fourier Test MSE")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Learning Curves: Raw vs Fourier")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "learning_curves.png"), dpi=200)
plt.show()

### (b)-(5)(6) 原圖與不同 Epoch 的重建結果

第一列：Raw coordinates；第二列：Fourier encoding。最左側為 Ground Truth。

In [ ]:
fig, axes = plt.subplots(2, len(SNAPSHOT_EPOCHS) + 1, figsize=(16, 7))

axes[0, 0].imshow(img_arr)
axes[0, 0].set_title("Ground Truth")
axes[0, 0].axis("off")

axes[1, 0].imshow(img_arr)
axes[1, 0].set_title("Ground Truth")
axes[1, 0].axis("off")

for j, epoch in enumerate(SNAPSHOT_EPOCHS, start=1):
    raw_img = np.clip(snap_raw[epoch].reshape(H, W, 3), 0.0, 1.0)
    enc_img = np.clip(snap_enc[epoch].reshape(H, W, 3), 0.0, 1.0)

    axes[0, j].imshow(raw_img)
    axes[0, j].set_title(f"Raw Epoch {epoch}")
    axes[0, j].axis("off")

    axes[1, j].imshow(enc_img)
    axes[1, j].set_title(f"Fourier Epoch {epoch}")
    axes[1, j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "epoch_reconstructions.png"), dpi=200)
plt.show()

## (c) Raw coordinates vs. Fourier encoding

In [ ]:
raw_final = np.clip(snap_raw[EPOCHS].reshape(H, W, 3), 0.0, 1.0)
enc_final = np.clip(snap_enc[EPOCHS].reshape(H, W, 3), 0.0, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img_arr)
axes[0].set_title("Ground Truth")

axes[1].imshow(raw_final)
axes[1].set_title(f"Raw (Test RMS={test_rms_raw:.4f})")

axes[2].imshow(enc_final)
axes[2].set_title(f"Fourier (Test RMS={test_rms_enc:.4f})")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "raw_vs_fourier.png"), dpi=200)
plt.show()

### 結果比較與討論

> **請在跑完實驗後，依據自己的實際數字與圖片修改下方文字（標示 ＿＿ 處）。**

**1. 觀察**

| 輸入 | Train RMS | Test RMS |
|---|---|---|
| Raw $(x, y)$ | ＿＿ | ＿＿ |
| Fourier features | ＿＿ | ＿＿ |

Raw coordinates 的重建圖＿＿（例如：偏模糊，邊緣、齒輪與細線等高頻細節不清楚，早期 epoch 還出現橫直條紋）；
Fourier encoding 在＿＿ epoch 就能還原出主要輪廓，最終的細節也較銳利，RMS 也＿＿（較低 / 較高）。

**2. 為什麼編碼能還原細節（Spectral bias）**

- 神經網路以梯度下降訓練時，傾向**先學習低頻成分，高頻成分收斂非常慢**，這稱為 *spectral bias*。
- 原始輸入 $(x, y)$ 是很平滑的線性變數，ReLU 網路若要組合出陡峭的邊緣（例如白底與深藍的交界），需要更多層、更多單元與更久的訓練，因此容易得到模糊的結果。
- Fourier feature mapping 把輸入改成 $[x,\ \sin(2^k\pi x),\ \cos(2^k\pi x)]$，$k=0,\dots,5$（$y$ 同理）。這等於在輸入端**直接提供多個頻率的基底函數**，網路只需對這些特徵做線性或淺層的非線性組合，就能表示高頻變化，不必從低頻慢慢「長」出高頻。
- 頻率以 $2^k$ 倍增，涵蓋由粗到細的多個尺度，所以大輪廓與小紋理都能被表示。
- 從 NTK（neural tangent kernel）的角度，這樣的編碼會拓寬核函數的有效頻寬，高頻成分的學習速度因而大幅提升。

**3. 補充**

若 Fourier 版本的 Train RMS 明顯低於 Test RMS，表示頻率過高時可能出現過擬合（測試像素帶有雜訊），可以調整 `NUM_FREQ` 在細節與泛化之間取捨。

In [ ]:
results_path = os.path.join(OUTPUT_DIR, "regression_results.csv")

with open(results_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)

    writer.writerow([
        "Experiment", "Input Dimension", "Hidden Layers", "Hidden Units",
        "Epochs", "Learning Rate", "Batch Size",
        "Train MSE", "Test MSE", "Train RMS", "Test RMS"
    ])

    writer.writerow([
        "Raw", 2, 3, HIDDEN_UNITS, EPOCHS, LEARNING_RATE, BATCH_SIZE,
        calculate_mse(model_raw, X_train, Y_train),
        calculate_mse(model_raw, X_test, Y_test),
        train_rms_raw, test_rms_raw
    ])

    writer.writerow([
        "Fourier", enc_dim, 3, HIDDEN_UNITS, EPOCHS, LEARNING_RATE, BATCH_SIZE,
        calculate_mse(model_enc, X_train_enc, Y_train),
        calculate_mse(model_enc, X_test_enc, Y_test),
        train_rms_enc, test_rms_enc
    ])

print("========== FINAL SUMMARY ==========")
print(f"Raw      Train RMS: {train_rms_raw:.6f}")
print(f"Raw      Test RMS : {test_rms_raw:.6f}")
print(f"Fourier  Train RMS: {train_rms_enc:.6f}")
print(f"Fourier  Test RMS : {test_rms_enc:.6f}")

print("\nSaved output files in:", OUTPUT_DIR)
print("- learning_curves.png")
print("- epoch_reconstructions.png")
print("- raw_vs_fourier.png")
print("- regression_results.csv")